In [1]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

In [18]:
import pandas as pd

import src
from src.load import DataLoader

r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [3]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

here() starts at /mnt/hdd/git/ytpop


# Load Data

In [10]:
dl = DataLoader()

videos = (
    dl.channels()
    .join(dl.videos(filtered=True, _ignore_sentence_filter=True), "channel_id")
    .to_pandas()
)
sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

/nix/store/299jaglw5pjxd3mrf780a3v9578sz1vz-python3-3.11.10-env/lib/python3.11/site-packages/ibis/expr/types/relations.py:685: FutureWarning: Selecting/filtering arbitrary expressions in `Table.__getitem__` is deprecated and will be removed in version 10.0. Please use `Table.select` or `Table.filter` instead.
  warnings.warn(


In [26]:
videos[videos.channel == "AfD TV"].head()

,channel_uploader_id,channel_follower_count,channel_collected,channel_description,channel,channel_id,channel_url,video_id,channel_right,video_title,video_duration,video_view_count,video_like_count,video_comment_count,video_was_live,video_description,video_datetime_upload
11422,@AfDTV,320000,2025-02-06 10:01:22,Offizieller YouTube-Kanal der Alternative für ...,AfD TV,UCq2rogaxLtQFrYG3X3KYNww,https://www.youtube.com/channel/UCq2rogaxLtQFr...,WBN424zjtH4,@AfDTV,Sprit-Preise runter? Das geht nur mit Steuerse...,455,9601,1814.0,NaN,False,Die Forderung des FDP-Vorsitzenden Lindner nac...,2022-03-22 13:00:28
11423,@AfDTV,320000,2025-02-06 10:01:22,Offizieller YouTube-Kanal der Alternative für ...,AfD TV,UCq2rogaxLtQFrYG3X3KYNww,https://www.youtube.com/channel/UCq2rogaxLtQFr...,GqfGNr48bkA,@AfDTV,Impfpflicht? Eine Art Erpressung! | 7 Tage Deu...,2478,18840,2637.0,NaN,False,AfD-Bundessprecher Tino Chrupalla (02:08) und ...,2022-03-18 13:53:00
11424,@AfDTV,320000,2025-02-06 10:01:22,Offizieller YouTube-Kanal der Alternative für ...,AfD TV,UCq2rogaxLtQFrYG3X3KYNww,https://www.youtube.com/channel/UCq2rogaxLtQFr...,xlXIez5FTdc,@AfDTV,Misswirtschaft der Ampel: Dem Mittelstand reic...,1447,10394,1378.0,NaN,False,Frequenz: Freiheit – der Podcast der AfD | Aus...,2023-11-24 15:38:59
11425,@AfDTV,320000,2025-02-06 10:01:22,Offizieller YouTube-Kanal der Alternative für ...,AfD TV,UCq2rogaxLtQFrYG3X3KYNww,https://www.youtube.com/channel/UCq2rogaxLtQFr...,BrtpiiYrJv8,@AfDTV,Volker Schnurrbusch: Geldverschwendung der EU ...,435,2338,276.0,NaN,False,Volker Schnurrbusch kandidiert bei der Europaw...,2023-08-06 10:51:41
11426,@AfDTV,320000,2025-02-06 10:01:22,Offizieller YouTube-Kanal der Alternative für ...,AfD TV,UCq2rogaxLtQFrYG3X3KYNww,https://www.youtube.com/channel/UCq2rogaxLtQFr...,B86MYjRiCSw,@AfDTV,Interview mit Kay Gottschalk nach der Wahl zum...,173,245,NaN,NaN,False,Erst wurde der AfD-Bundestagsabgeordnete Kay G...,2017-12-08 00:25:40


# Dataset Summary Table

In [35]:
channel_overview = (
    videos.merge(sents, on="video_id", how="left")
    .groupby("channel", observed=True)
    .agg(
        ch_followers=("channel_follower_count", "first"),
        videos=("video_was_live", lambda x: (x == 0).sum()),
        livestreams=("video_was_live", "sum"),
        n_sentences=("n_sents", "sum"),
        avg_likes=("video_like_count", "mean"),
        avg_views=("video_view_count", "mean"),
        avg_duration=("video_duration", "mean"),
        # avg_comments=("video_comment_count", lambda x: x.dropna().mean()),
        # n_disabled_comments=("video_comment_count", lambda x: x.isna().sum()),
        first_video=("video_datetime_upload", "min"),
        latest_video=("video_datetime_upload", "max"),
    )
)

In [36]:
channel_overview

,ch_followers,videos,livestreams,n_sentences,avg_likes,avg_views,avg_duration,first_video,latest_video
channel,,,,,,,,,
AfD BT,515000,6243,527,344843.0,4407.3,54929.7,1682.2,2017-12-06 13:23:54,2025-02-04 18:00:08
AfD TV,320000,1969,156,162171.0,4413.7,56527.5,1773.6,2017-12-08 00:21:22,2025-02-03 15:19:09
CDU,28400,957,239,77001.0,136.5,16744.9,1692.8,2017-12-11 16:28:36,2025-02-04 22:08:58
CSU,6340,179,250,12653.0,40.9,9975.1,2745.1,2017-12-14 21:19:06,2025-02-04 04:28:14
FDP,26900,851,117,52737.0,82.0,25399.2,1063.8,2018-01-06 16:02:32,2025-02-04 17:56:50
Greens,32900,644,91,49051.0,153.2,13531.1,1193.5,2018-01-12 10:16:37,2025-02-04 20:52:23
Left,72900,779,408,128387.0,666.0,12059.8,1521.0,2017-12-11 14:33:45,2025-02-04 18:06:03
SPD,31600,1111,213,156407.0,151.7,8110.7,1778.4,2017-12-07 12:17:56,2025-02-03 11:34:41


In [ ]:
channel_overview

,ch_videos,ch_followers,n_sentences,livestreams,avg_likes,avg_views,avg_duration,first_video,latest_video
channel,,,,,,,,,
AfD BT,6770,515000,344843.0,527,4407.3,54929.7,1682.2,2017-12-06 13:23:54,2025-02-04 18:00:08
AfD TV,2125,320000,162171.0,156,4413.7,56527.5,1773.6,2017-12-08 00:21:22,2025-02-03 15:19:09
CDU,1196,28400,77001.0,239,136.5,16744.9,1692.8,2017-12-11 16:28:36,2025-02-04 22:08:58
CSU,429,6340,12653.0,250,40.9,9975.1,2745.1,2017-12-14 21:19:06,2025-02-04 04:28:14
FDP,968,26900,52737.0,117,82.0,25399.2,1063.8,2018-01-06 16:02:32,2025-02-04 17:56:50
Greens,735,32900,49051.0,91,153.2,13531.1,1193.5,2018-01-12 10:16:37,2025-02-04 20:52:23
Left,1187,72900,128387.0,408,666.0,12059.8,1521.0,2017-12-11 14:33:45,2025-02-04 18:06:03
SPD,1324,31600,156407.0,213,151.7,8110.7,1778.4,2017-12-07 12:17:56,2025-02-03 11:34:41


In [ ]:
channel_overview

,ch_videos,ch_followers,n_sentences,livestreams,avg_likes,avg_views,avg_duration,avg_comments,first_video,latest_video
channel,,,,,,,,,,
AfD BT,6097,515000,344843,0,4299.9,52748.4,414.9,418.7,2017-12-06 13:23:54,2025-01-31 15:47:51
AfD TV,1740,320000,162171,0,4277.6,51272.7,580.2,NaN,2017-12-08 00:21:22,2025-01-24 18:30:05
CDU,767,28400,77001,0,146.7,10631.2,597.6,54.0,2017-12-11 16:28:36,2025-01-26 12:27:06
CSU,161,6340,12653,0,40.7,22498.6,460.6,20.4,2017-12-14 21:19:06,2025-01-12 10:00:06
FDP,640,26900,52737,0,105.0,15015.1,618.5,1.2,2018-01-06 16:02:32,2025-01-27 19:05:52
Greens,588,32900,49051,0,129.9,10164.7,639.3,145.2,2018-01-27 10:45:39,2025-01-31 18:46:45
Left,658,72900,128387,0,964.9,18119.5,623.7,NaN,2017-12-11 14:33:45,2025-01-31 18:03:12
SPD,921,31600,156407,0,161.5,9127.7,667.6,42.7,2017-12-07 12:17:56,2025-01-31 15:25:04


In [ ]:
channel_overview

,ch_videos,ch_followers,n_sentences,n_elite,n_pplcentr,avg_likes,avg_views,avg_duration,avg_comments,first_video,latest_video
channel,,,,,,,,,,,
AfD BT,5215,388000,295255,39747,5984,3861.4,45749.6,436.5,397.6,2017-12-06,2024-01-20
AfD TV,1442,250000,142810,17056,3592,3652.1,43601.4,666.1,398.4,2017-12-07,2024-01-19
CDU,613,21900,53017,918,1419,67.7,9165.2,661.3,43.1,2017-12-11,2024-01-19
CSU,141,5170,10337,286,245,36.1,22944.1,453.4,7.7,2017-12-14,2023-10-05
FDP,461,23300,36094,1403,886,0.5,5580.9,670.5,0.7,2018-01-06,2024-01-06
Greens,455,26100,45667,1415,1314,75.8,4459.9,901.3,0.2,2018-01-27,2023-12-13
Left,429,29000,45912,2420,1374,260.0,10734.1,878.3,46.3,2017-12-11,2024-01-17
SPD,469,24200,65523,1380,2163,100.5,5204.8,1133.0,25.7,2017-12-07,2024-01-18


In [10]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 1481.54 hours


In [11]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 9225


In [12]:
# number of sentencs

count_sents = channel_overview.n_sentences.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 694615


In [13]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T

summary_table

channel,AfD BT,AfD TV,CDU,CSU,FDP,Greens,Left,SPD
ch_videos,5215.0,1442.0,613.0,141.0,461.0,455.0,429.0,469.0
ch_followers,388000.0,250000.0,21900.0,5170.0,23300.0,26100.0,29000.0,24200.0
n_sentences,295255.0,142810.0,53017.0,10337.0,36094.0,45667.0,45912.0,65523.0
n_elite,39747.0,17056.0,918.0,286.0,1403.0,1415.0,2420.0,1380.0
n_pplcentr,5984.0,3592.0,1419.0,245.0,886.0,1314.0,1374.0,2163.0
avg_likes,3861.4,3652.1,67.7,36.1,0.5,75.8,260.0,100.5
avg_views,45749.6,43601.4,9165.2,22944.1,5580.9,4459.9,10734.1,5204.8
avg_duration,436.5,666.1,661.3,453.4,670.5,901.3,878.3,1133.0
avg_comments,397.6,398.4,43.1,7.7,0.7,0.2,46.3,25.7


In [14]:
path = src.OUT / "tables/dataset_summary.csv"
summary_table.to_csv(path)

# View Count Violin Plot

In [15]:
df = videos.merge(sents, on="video_id")

In [16]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=5)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")


ggarrange(view_plot, like_plot, ncol=2)

ggsave(here(r_out, "/figures/view_count.svg"))

Saving 13.9 x 8.33 in image


# Populism Amount Plot

In [41]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
)

elite_plot = ggplot(df_plot, aes(x=channel, y=elite, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% Anti-Elitism")

pplcentr_plot = ggplot(df_plot, aes(x=channel, y=pplcentr, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% People-Centrism")


ggarrange(elite_plot, pplcentr_plot, ncol=2)

ggsave(here(r_out, "/figures/populism_per_party.svg"))

Saving 13.9 x 8.33 in image
